# Factor analysis — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def pairwise_sq(A,B):
    A=np.asarray(A,float); B=np.asarray(B,float)
    return ((A[:,None,:]-B[None,:,:])**2).sum(axis=2)
def softmax(z):
    z=np.asarray(z,float); z=z-np.max(z)
    e=np.exp(z); return e/e.sum()
def entropy(p):
    p=np.asarray(p,float); p=p[p>0]
    return float(-(p*np.log(p)).sum())
def center(K):
    n=K.shape[0]; H=np.eye(n)-np.ones((n,n))/n
    return H@K@H
print('setup ok')

## One focused concept: the core computation inside Factor analysis
These five cells isolate the base calculation, a knob turn, a contrast, an edge case, and a tiny end-to-end pass for Factor analysis.

In [ ]:
# 1) center a data matrix before extracting structure
X=np.array([[1.,2.],[2.,1.],[3.,4.],[4.,3.]])
Xc=X-X.mean(axis=0)
plt.figure(figsize=(5,4)); plt.scatter(Xc[:,0],Xc[:,1],s=80); plt.axhline(0,c='gray'); plt.axvline(0,c='gray'); plt.title('Factor analysis: centered data')
assert np.allclose(Xc.mean(axis=0),0)

In [ ]:
# 2) singular values/eigenvalues show how much structure each direction carries
X=np.array([[1.,2.],[2.,1.],[3.,4.],[4.,3.]],float); Xc=X-X.mean(axis=0)
U,S,Vt=np.linalg.svd(Xc,full_matrices=False)
plt.figure(figsize=(5,3)); plt.bar([1,2],S**2); plt.title('Factor analysis: component energy')
assert S[0]>S[1]

In [ ]:
# 3) one-dimensional projection keeps the strongest direction
X=np.array([[1.,2.],[2.,1.],[3.,4.],[4.,3.]],float); Xc=X-X.mean(axis=0)
U,S,Vt=np.linalg.svd(Xc,full_matrices=False); z=Xc@Vt[:1].T; rec=z@Vt[:1]
plt.figure(figsize=(5,4)); plt.scatter(Xc[:,0],Xc[:,1],label='centered'); plt.scatter(rec[:,0],rec[:,1],marker='x',s=90,label='rank-1'); plt.legend(); plt.title('Factor analysis: projection/reconstruction')
assert rec.shape==Xc.shape and np.linalg.norm(Xc-rec)<np.linalg.norm(Xc)

In [ ]:
# 4) reconstruction error falls as rank increases
X=np.array([[1.,2.],[2.,1.],[3.,4.],[4.,3.]],float); Xc=X-X.mean(axis=0); U,S,Vt=np.linalg.svd(Xc,full_matrices=False)
errs=[]
for k in [0,1,2]:
    rec=(U[:,:k]*S[:k])@Vt[:k,:] if k else np.zeros_like(Xc); errs.append(np.linalg.norm(Xc-rec))
plt.figure(figsize=(5,3)); plt.plot([0,1,2],errs,'-o'); plt.xlabel('rank/components'); plt.ylabel('error'); plt.title('Factor analysis: more components fit better')
assert errs[2]<1e-10 and errs[1]<errs[0]

In [ ]:
# 5) transformed coordinates are lower-dimensional features
X=np.array([[1.,2.],[2.,1.],[3.,4.],[4.,3.]],float); Xc=X-X.mean(axis=0); U,S,Vt=np.linalg.svd(Xc,full_matrices=False); Z=Xc@Vt.T
plt.figure(figsize=(5,4)); plt.scatter(Z[:,0],Z[:,1],s=80); plt.axhline(0,c='gray'); plt.axvline(0,c='gray'); plt.title('Factor analysis: learned coordinate system')
assert Z.shape==(4,2) and np.allclose(np.var(Z,axis=0),S**2/4)